In [1]:
#other utilities and packages
import torch
import pandas as pd
from pathlib import Path
import numpy as np
import pandas as pd
import random
from glob import glob
import sklearn

from tqdm.autonotebook import tqdm
from sklearn.metrics import average_precision_score, roc_auc_score
from pathlib import Path

#set up plotting
from matplotlib import pyplot as plt
plt.rcParams['figure.figsize']=[15,5] #for large visuals
%config InlineBackend.figure_format = 'retina'

# opensoundscape transfer learning tools
from opensoundscape.ml.shallow_classifier import MLPClassifier, fit, fit_classifier_on_embeddings
import opensoundscape as opso
import bioacoustics_model_zoo as bmz


# load the annotation file from Raven
annotation_file = Path(r"/mnt/c/Users/247404/Documents/2024/HABITWAL FINAL REPORT/Airgun_detector/annotations_airguns150.txt")
audio_file = Path(r"/mnt/j/HABITATWal/AMAR/HABITATWal.RDS/Sound Catalog AMARs/Airgun/all_together.wav") 


/tmp/ipykernel_1046893/1445056552.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [2]:
# let's generate clip labels of 5s duration (to match Perch input duration) using the raven annotations
# and some utility functions from opensoundscape
from opensoundscape.annotations import BoxedAnnotations


# update the paths to where we have the audio and raven files stored


annotations = BoxedAnnotations.from_raven_files(
    raven_files=[annotation_file],
    audio_files=[audio_file],
    annotation_column=None,
)
annotations.df["annotation"] = "airgun"
# generate labels for 5s clips, including any labels that overlap by at least 0.2 seconds
# Parameters to use for label creation
clip_duration = 5
clip_overlap = 0
min_label_overlap = 0.25

# Create dataframe of one-hot labels
clip_labels = annotations.clip_labels(
    clip_duration = clip_duration,
    clip_overlap = clip_overlap,
    min_label_overlap = min_label_overlap,
)


In [3]:
clip_labels.sum()

airgun    201
dtype: int64

In [4]:
labels = clip_labels.sort_index(level="start_time")
n = len(labels)
train_df = labels.iloc[: int(0.5 * n)]
val_df   = labels.iloc[int(0.5 * n) : int(0.85 * n)]
test_df  = labels.iloc[int(0.85 * n) :]

for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(name, len(d), "clips,",d["airgun"].sum(), "with airgun")

train 151 clips, 81 with airgun
val 106 clips, 83 with airgun
test 46 clips, 37 with airgun


In [5]:
perch = bmz.Perch2ONNX(headless=True)


In [6]:
train_emb=np.asanyarray(perch.embed(train_df, return_dfs=False, return_preds=False))
val_emb=perch.embed(val_df, return_dfs=False, return_preds=False)

/home/biofelip/anaconda3/envs/cv4animals/lib/python3.10/site-packages/opensoundscape/ml/cnn.py:2930: UserWarning: The columns of input samples df differ from `model.classes`. Discarding sample df columns.
  warnings.warn(


  0%|          | 0/151 [00:00<?, ?it/s]

/home/biofelip/anaconda3/envs/cv4animals/lib/python3.10/site-packages/opensoundscape/ml/cnn.py:2930: UserWarning: The columns of input samples df differ from `model.classes`. Discarding sample df columns.
  warnings.warn(


  0%|          | 0/106 [00:00<?, ?it/s]

In [16]:
val_emb

array([[-0.0562857 ,  0.10139447,  0.14775863, ...,  0.09830336,
         0.14518154,  0.09135054],
       [-0.01032408,  0.15870894,  0.2966455 , ..., -0.01667019,
         0.08969024, -0.02154506],
       [-0.03610438,  0.14449045,  0.23220737, ...,  0.05206167,
         0.2691943 ,  0.02454892],
       ...,
       [ 0.01159446,  0.07831611, -0.03340176, ...,  0.07342487,
         0.01634364,  0.08854495],
       [-0.02923908,  0.08133911, -0.05756274, ...,  0.04178564,
         0.11120035,  0.09489386],
       [ 0.00445436, -0.02487382,  0.10702331, ...,  0.06945001,
        -0.08263015,  0.01642956]], shape=(106, 1536), dtype=float32)

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score



In [19]:
# 3. Small classifier on top
clf = LogisticRegression(max_iter=5000, class_weight="balanced")   # balanced: you have few negatives
clf.fit(train_emb, train_df['airgun'])

val_scores = clf.predict_proba(val_emb)[:, 1]
print("val AUC:", roc_auc_score(val_df["airgun"], val_scores))


val AUC: 0.8638030382399162


Here we will save the generated classifier based on perch. Interestingly enough it did not do too bad with 86% accuracy. 

In [20]:
import joblib
joblib.dump(clf, "perch2onnx_headless_5s_logreg.joblib")   # save

# clf = joblib.load("perch2onnx_headless_5s_logreg.joblib")  # load later

['perch2onnx_headless_5s_logreg.joblib']